This notebooks serves as a testing playground for the cxllin/Llama2-7b-med-v1 LLM. This model is very small, but has been finetuned on medical data.

Author: Henri Smidt

Email: finn.smidt@stud.uni-heidelberg.de


In [9]:
%pip install --upgrade --quiet  xformers --quiet #TODO: try out xformer as this is supposed to be more memory efficient
%pip install --upgrade --quiet  langchain    --quiet
%pip install --upgrade --quiet  bitsandbytes --quiet
%pip install --upgrade --quiet  python-dotenv --quiet
%pip install accelerate --quiet



NotImplementedError: A UTF-8 locale is required. Got ANSI_X3.4-1968

In [4]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import os
from dotenv import load_dotenv
import transformers
from torch import cuda, bfloat16


# load environment variables from .env file
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

bitsAndBites_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16,
)

# hf = HuggingFacePipeline.from_model_id(
#     model_id="cxllin/Llama2-7b-med-v1",
#     task="text-generation",
#     pipeline_kwargs={"max_new_tokens": 10
#                      trust_remote_code=True,
#                      quantization_config=bitsAndBites_config,
#                      device_map='auto',
#                      },
# )


# OR


model_id = "cxllin/Llama2-7b-med-v1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bitsAndBites_config,
    device_map="auto",
    do_sample=True,
    token=os.environ.get("HF_AUTH"),
)

model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:381: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:386: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )


In [5]:
model.get_memory_footprint()

3829940224

In [15]:
pipe = pipeline(
    task="text-generation",
    model=model,
    return_full_text=True,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.01,
    repetition_penalty=1.1,
)

hf = HuggingFacePipeline(pipeline=pipe)

In [23]:
from langchain import PromptTemplate, LLMChain

template = """You are an assistant for medical question-answering tasks. You must use the following two pieces of retrieved context to answer the question. Cite the retrieved context you used to answer the question. If none of the retrieved contexts provide the answer, say that you couldnt find any relevant sources and try to answer the answer without the retrieved documents. However, if you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""  # Not using the context.

template = """Generate an answer for the following question. the knowledge you can use can only be derived from the two contexts, that are given. Do not make up the answer yourself.
Question: {question}
Context: {context} """  # Using the context, but not consistently. Sometimes it hallucinates.

template = """Generate an answer for the following question using only these two contexts as your base of knowledge (dont make up your own answer): {context} Say which of the sources you used. Do not state why you used which source.
Question: {question}
"""  # Answer is too short

prompt = PromptTemplate(input_variables=["question", "context"], template=template)

chain = LLMChain(prompt=prompt, llm=hf)

question = "What is electroencephalography?"
context = """Source: Wikipedia: Dogs like to run around and play.
Source: Journal of Medicine: Electroencephalography is a method to measure the weight of a patients feet.
"""

print(chain.predict(question=question, context=context))

Answer: The second source. 
